## Action Genome / STTran — Dataset & prediction statistics

This notebook is a **single entry point** for the quantitative summaries we use in the project. Read it top to bottom, or jump to a section heading. Every plot is produced from **saved terminal logs** (parsed scene graphs per frame); you do **not** need to re-run the detector here.

### What you will find

- **Section 1 — Frequencies** — bar charts: frame-normalized presence vs assignment-normalized shares (predicates, objects, and predicate groups `@` / `^` / `+`).
- **Section 1 — Lorenz curves** — same underlying counts as the bars, shown as cumulative concentration (with **Gini**); useful for arguing *how* imbalanced the tail is.
- **Section 1b — Graph structure** — undirected 0/1 adjacency per frame: degree by node class + simple spectral summaries (CSV + quick histograms).
- **Section 2 — Confidence** — boxplots of softmax scores per predicate (by group) + exported CSV/JSON stats.
- **Section 3 — Thresholds** — example rules `μ + kσ` on pooled scores (same parser settings as above).
- **Section 4 — GT sanity** — deep scan of `person_bbox.pkl` for multi-person frames.
- **Section 5 — Per-video** — optional grid of `edge_evolution.png` under each video folder.

### Before you start

- Default inputs live under `STTran/output/logs/first5_videos/` (change `LOGS_DIR` in the setup cell if you analyze another run).
- After pulling git changes to plotting code, use **Kernel → Restart & Run All** so `sys.path`, Matplotlib style, and embedded figures stay in sync.


In [1]:
import os
import sys
from pathlib import Path

# --- Matplotlib backend (inline figures in the notebook) ---
%matplotlib inline

# --- Repo paths: run from project root or from inside STTran ---
REPO_ROOT = Path.cwd().parent if (Path.cwd().name == "STTran") else Path.cwd()
STTRAN_DIR = REPO_ROOT / "STTran"
sys.path.insert(0, str(STTRAN_DIR))  # so ``plot_log_frequencies`` et al. import from this clone

# --- Global plot style (matches the Agg scripts on disk) ---
try:
    from plot_matplotlib_style import apply_notebook_style
    apply_notebook_style()
except Exception as _e:
    print("(plot style skipped)", _e)

LOGS_DIR = STTRAN_DIR / "output" / "logs" / "first5_videos"
OUT_ROOT = STTRAN_DIR / "output" / "first5_videos"
SUMMARY_DIR = OUT_ROOT / "summary_plots"

print("REPO_ROOT:", REPO_ROOT)
print("LOGS_DIR:", LOGS_DIR)
print("OUT_ROOT:", OUT_ROOT)
print("SUMMARY_DIR:", SUMMARY_DIR)

assert LOGS_DIR.exists(), f"Missing logs dir: {LOGS_DIR}"


REPO_ROOT: /Users/tommasoaiello/Desktop/Magistrale/Secondo Semestre/Computer Vision/CV_Project
LOGS_DIR: /Users/tommasoaiello/Desktop/Magistrale/Secondo Semestre/Computer Vision/CV_Project/STTran/output/logs/first5_videos
OUT_ROOT: /Users/tommasoaiello/Desktop/Magistrale/Secondo Semestre/Computer Vision/CV_Project/STTran/output/first5_videos
SUMMARY_DIR: /Users/tommasoaiello/Desktop/Magistrale/Secondo Semestre/Computer Vision/CV_Project/STTran/output/first5_videos/summary_plots


In [2]:
# Shared parser: identical to ``plot_log_frequencies.py`` / ``export_graphs_json.py``.
from viz_terminal_scene_graphs import parse_terminal_log  # noqa: F401

log_paths = sorted(LOGS_DIR.glob("*.log"))
print("#logs:", len(log_paths))
print("first 5:", [p.name for p in log_paths[:5]])

# TOPK limits must match how the ``.log`` files were produced (see ``run_first5_videos_all_frames.py``).
TOPK_S = int(os.environ.get("TOPK_SPATIAL", "4"))
TOPK_C = int(os.environ.get("TOPK_CONTACT", "4"))
print("TOPK_S:", TOPK_S, "TOPK_C:", TOPK_C)


#logs: 38
first 5: ['00607.mp4.log', '00T1E.mp4.log', '015XE.mp4.log', '01THT.mp4.log', '02DPI.mp4.log']
TOPK_S: 4 TOPK_C: 4


## 1) Frequencies and Lorenz imbalance curves

All plotting logic lives in `STTran/plot_log_frequencies.py` (the notebook only calls it).

### Bar charts

- **Frame-normalized** — a frame contributes at most once per predicate (or object class) if that label appears anywhere in that frame’s graph.
- **Assignment-normalized** — each edge or node occurrence counts; denominators are total edges (optionally split by `@` / `^` / `+`) or total nodes.

### Lorenz curves (2×2 figure)

For each family of counts we sort categories from **rarest to most frequent**. The **x** axis is the cumulative fraction of **category labels**; the **y** axis is the cumulative fraction of **total count** (frame presences or assignments). The dashed diagonal is perfect equality; a curve that bends **far below** it means a few categories hold most of the mass. The legend reports a discrete **Gini** coefficient in `[0, 1]` (higher ⇒ more concentrated).


In [ ]:
from pathlib import Path

# Re-apply style after editing plotting code (avoids restarting the kernel).
try:
    from plot_matplotlib_style import apply_notebook_style
    apply_notebook_style()
except Exception:
    pass

from plot_log_frequencies import (
    generate_assignment_normalized_plots,
    generate_frequency_plots,
    generate_lorenz_plots,
    generate_predicate_frequency_plots_by_group,
)

SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
_tmp = SUMMARY_DIR

# Bar charts (same entry points as ``python plot_log_frequencies.py``).
pred_png, obj_png = generate_frequency_plots(
    logs_dir=str(LOGS_DIR), out_dir=str(_tmp), topk_spatial=TOPK_S, topk_contact=TOPK_C
)
att_png, spa_png, con_png = generate_predicate_frequency_plots_by_group(
    logs_dir=str(LOGS_DIR), out_dir=str(_tmp), topk_spatial=TOPK_S, topk_contact=TOPK_C
)
edge_all, edge_att, edge_spa, edge_con, node_obj = generate_assignment_normalized_plots(
    logs_dir=str(LOGS_DIR), out_dir=str(_tmp), topk_spatial=TOPK_S, topk_contact=TOPK_C
)

# Lorenz + Gini on the same raw counters as the bars (see markdown above).
lorenz_png = generate_lorenz_plots(
    logs_dir=str(LOGS_DIR), out_dir=str(_tmp), topk_spatial=TOPK_S, topk_contact=TOPK_C
)

# Inline previews: embed bytes so the UI does not cache stale PNGs by path.
from IPython.display import Image as IPyImage, display, Markdown

for label, pth in [
    ("Predicate · frame-normalized", pred_png),
    ("Object class · frame-normalized", obj_png),
    ("Attention @ · frame-normalized", att_png),
    ("Spatial ^ · frame-normalized", spa_png),
    ("Contact + · frame-normalized", con_png),
    ("All predicates · assignment-normalized", edge_all),
    ("Attention @ · assignment-normalized", edge_att),
    ("Spatial ^ · assignment-normalized", edge_spa),
    ("Contact + · assignment-normalized", edge_con),
    ("Object class · node-normalized", node_obj),
    ("Lorenz curves · label imbalance (2×2)", lorenz_png),
]:
    data = Path(pth).read_bytes()
    display(Markdown(f"### {label}"))
    display(IPyImage(data=data, format="png", embed=True))

(pred_png, obj_png, att_png, spa_png, con_png, edge_all, edge_att, edge_spa, edge_con, node_obj, lorenz_png)


In [ ]:
# Optional: list every PNG under SUMMARY_DIR (frequencies, Lorenz, boxplots, …).
# Re-run after Sections 1 and 2 so moved/copied figures show up here.
from pathlib import Path
from IPython.display import Image as IPyImage, display, Markdown

pngs = sorted(SUMMARY_DIR.glob("*.png"))
print("PNG gallery in", SUMMARY_DIR, "→", len(pngs), "files")

pngs = sorted(pngs, key=lambda p: ("boxplot" in p.name, p.name))
if not pngs:
    print("(none yet — run Sections 1 and 2 first)")
else:
    for p in pngs:
        display(Markdown(f"**`{p.name}`**"))
        display(IPyImage(data=Path(p).read_bytes(), format="png", embed=True))


## 1b) Graph structure from prediction logs (undirected / unweighted)

We reuse the **same** `parse_terminal_log` as the frequency scripts. Per frame we build:

- **Undirected binary adjacency** \(A\): \(A_{ij}=1\) iff **any** predicted edge (attention / spatial / contact) links nodes \(i\) and \(j\). Scores and predicate names are ignored — only existence matters.
- **Degree by node category** — undirected degree of each node, grouped by its class string `cls` (one row per node occurrence across frames).
- **Spectral summaries** (real symmetric matrices via `numpy.linalg.eigvalsh`):
  - **Adjacency gap** = \(\lambda_{\max}(A) - \lambda_{2}(A)\) with eigenvalues sorted ascending (two largest).
  - **Normalized Laplacian** \(L = I - D^{-1/2} A D^{-1/2}\) with the usual isolated-vertex convention (\(L_{ii}=1\), off-diagonals 0). We report \(\lambda_2(L)\) (algebraic connectivity for a connected graph; disconnected graphs often have \(\lambda_2 = 0\)).

**CSV outputs:** `summary_plots/node_degree_by_class.csv` and `summary_plots/spectral_per_frame.csv` (one row per parsed frame — the latter can be large).

**TOPK:** keep `TOPK_S` / `TOPK_C` aligned with how the `.log` files were produced (defaults in `run_first5_videos_all_frames.py` are 4 / 4).


In [ ]:
# Aggregate graph metrics across all logs, then save CSVs and show quick summaries.
from graph_structure_analysis import (
    aggregate_degrees_by_class,
    iter_spectral_rows,
    spectral_summary,
    save_degree_table,
    save_spectral_rows,
)

deg_agg = aggregate_degrees_by_class(log_paths, topk_spatial=TOPK_S, topk_contact=TOPK_C)
save_degree_table(deg_agg, SUMMARY_DIR / "node_degree_by_class.csv")

spec_rows = list(iter_spectral_rows(log_paths, topk_spatial=TOPK_S, topk_contact=TOPK_C))
save_spectral_rows(spec_rows, SUMMARY_DIR / "spectral_per_frame.csv")

print("dataset spectral summary:", spectral_summary(spec_rows))

import pandas as pd

deg_df = pd.read_csv(SUMMARY_DIR / "node_degree_by_class.csv").sort_values("count_nodes", ascending=False)
display(deg_df.head(30))

spec_df = pd.read_csv(SUMMARY_DIR / "spectral_per_frame.csv")
display(
    spec_df[["n_nodes", "n_edges_undirected", "adj_gap", "lap_algebraic_conn"]].describe(
        percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]
    )
)

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(spec_df["adj_gap"].dropna(), bins=40, color="#334155", alpha=0.85)
axes[0].set_title("Adjacency eigenvalue gap (λ_max − λ_2)")
axes[0].set_xlabel("gap")
axes[0].set_ylabel("frames")

axes[1].hist(spec_df["lap_algebraic_conn"].dropna(), bins=40, color="#0f766e", alpha=0.85)
axes[1].set_title("Normalized Laplacian λ₂ (algebraic connectivity)")
axes[1].set_xlabel("λ₂")
axes[1].set_ylabel("frames")
plt.tight_layout()
plt.show()


## 2) Predicate confidence — boxplots and table export

We call `STTran/plot_predicate_score_stats.py` as a subprocess so the notebook stays thin and **bit-identical** to the CLI. That script writes:

- three **boxplots** (attention / spatial / contact), and
- `predicate_confidence_stats.csv` + `.json` — empirical mean, std, quantiles, and counts **per predicate**.

Artifacts are moved into `SUMMARY_DIR` so everything lives beside the frequency plots.


In [ ]:
import subprocess
from pathlib import Path
from IPython.display import Image as IPyImage, display, Markdown

subprocess.check_call([sys.executable, str(STTRAN_DIR / "plot_predicate_score_stats.py")])

# Keep a single folder for all figures + stats used in this notebook.
for fname in [
    "predicate_confidence_stats.csv",
    "predicate_confidence_stats.json",
    "predicate_scores_attention_boxplot.png",
    "predicate_scores_spatial_boxplot.png",
    "predicate_scores_contact_boxplot.png",
]:
    p = OUT_ROOT / fname
    if p.exists():
        p.replace(SUMMARY_DIR / fname)

for title, fname in [
    ("Attention @ — score distribution", "predicate_scores_attention_boxplot.png"),
    ("Spatial ^ — score distribution", "predicate_scores_spatial_boxplot.png"),
    ("Contact + — score distribution", "predicate_scores_contact_boxplot.png"),
]:
    p = SUMMARY_DIR / fname
    if p.is_file():
        display(Markdown(f"### {title}"))
        display(IPyImage(data=Path(p).read_bytes(), format="png", embed=True))

print("stats:", SUMMARY_DIR / "predicate_confidence_stats.csv")


In [7]:
# Pool softmax scores by relation group (same parser + TOPK as Section 1).
import numpy as np

scores = {"att": [], "spatial": [], "contact": []}
for lp in log_paths:
    frames = parse_terminal_log(str(lp), topk_spatial=TOPK_S, topk_contact=TOPK_C)
    for fr in frames.values():
        for e in fr.edges:
            if e.group in scores:
                scores[e.group].append(float(e.score))

for g in ["att", "spatial", "contact"]:
    arr = np.asarray(scores[g], dtype=np.float32)
    print(g, "n_edges=", arr.size, "mean=", float(arr.mean()), "std=", float(arr.std()))


att n_edges= 4993 mean= 0.4626016318798065 std= 0.38560348749160767
spatial n_edges= 6158 mean= 0.2246021181344986 std= 0.0998462438583374
contact n_edges= 6158 mean= 0.10088177770376205 std= 0.039822421967983246


## 3) Example score thresholds

A simple per-group rule is a multiple of the empirical standard deviation above the mean:

- **tolerant:** \(\mu + 0.5\sigma\)
- **stricter:** \(\mu + 1.0\sigma\)

Here \(\mu\) and \(\sigma\) are the mean and std of **all** edge-level confidence scores in that group (attention / spatial / contact), recomputed in the previous cell.


In [8]:
import numpy as np

mu = {g: float(np.mean(scores[g])) for g in scores}
sd = {g: float(np.std(scores[g])) for g in scores}

th_05 = {g: mu[g] + 0.5 * sd[g] for g in scores}
th_10 = {g: mu[g] + 1.0 * sd[g] for g in scores}

print("mu:", mu)
print("std:", sd)
print("threshold mu+0.5std:", th_05)
print("threshold mu+1.0std:", th_10)


mu: {'att': 0.4626016422992189, 'spatial': 0.22460214355310165, 'contact': 0.10088177979863593}
std: {'att': 0.38560351013332933, 'spatial': 0.09984624038986688, 'contact': 0.03982242154240956}
threshold mu+0.5std: {'att': 0.6554033973658836, 'spatial': 0.2745252637480351, 'contact': 0.1207929905698407}
threshold mu+1.0std: {'att': 0.8482051524325482, 'spatial': 0.32444838394296854, 'contact': 0.1407042013410455}


## 4) GT person boxes — quick sanity scan

Action Genome stores GT person boxes in `dataset/ag/annotations/person_bbox.pkl`.
This cell counts how many person boxes appear per frame key and reports how many frames have **two or more** persons (useful to catch unexpected layout in the pickle).


In [9]:
import pickle
from collections import Counter
import numpy as np

AG_ROOT = REPO_ROOT / "dataset" / "ag"
PERSON_PKL = AG_ROOT / "annotations" / "person_bbox.pkl"

with open(PERSON_PKL, "rb") as f:
    person = pickle.load(f)

count_dist = Counter()
for k, v in person.items():
    b = v.get("bbox") if isinstance(v, dict) else v
    if b is None:
        n = 0
    else:
        arr = np.asarray(b)
        if arr.ndim == 2 and arr.shape[1] == 4:
            n = int(arr.shape[0])
        elif arr.ndim == 1 and arr.size >= 4:
            n = 1
        else:
            n = 0
    count_dist[n] += 1

print("Total frames in person_bbox.pkl:", sum(count_dist.values()))
print("Distribution (#persons -> #frames):")
for n in sorted(count_dist):
    print(f"  {n}: {count_dist[n]}")
print("Frames with >=2 persons:", sum(c for n, c in count_dist.items() if n >= 2))


/var/folders/np/yvh5ndg15b56jryxv8dqc_x00000gn/T/ipykernel_10748/585671641.py:9: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  person = pickle.load(f)


Total frames in person_bbox.pkl: 288782
Distribution (#persons -> #frames):
  0: 40482
  1: 248300
Frames with >=2 persons: 0


## 5) Per-video edge evolution thumbnails

For every processed video folder under `OUT_ROOT` that contains `edge_evolution.png`, we render a simple grid (no fixed cap; many videos means a large figure).

The next cell calls `plot_edge_evolution_grid.py` so subplot handling lives in one place and matches what you run from the repo.


In [ ]:
# Thumbnails are built in ``plot_edge_evolution_grid.py`` (single source of truth).
from plot_edge_evolution_grid import plot_edge_evolution_grid

plot_edge_evolution_grid(OUT_ROOT)
